# University Query Management System (Fixed & Working Version)

This notebook loads `university_query_train.csv` and `university_query_test.csv` (provided alongside this notebook) and fixes the issues found in the original version:
- The trained ML model is now actually used by the chatbot to predict priority (previously trained but never called).
- Department routing and auto-response logic are unified into one rule set (previously inconsistent between two separate functions).
- Word-boundary regex matching is used so `"exam"` doesn't wrongly match words like `"example"`.
- A train/test text-overlap check is added to flag inflated accuracy.
- Input validation added (empty/invalid queries handled gracefully).

**Place `university_query_train.csv` and `university_query_test.csv` in the same folder as this notebook before running.**

## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import re
import random

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

## 2. Load Data

In [4]:
train_df = pd.read_csv("university_query_train.csv")
test_df = pd.read_csv("university_query_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
train_df.head()

Train shape: (5000, 5)
Test shape: (1000, 5)


## 3. Exploratory Data Analysis

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Query_ID          5000 non-null   int64
 1   Student_Query     5000 non-null   str  
 2   Department        5000 non-null   str  
 3   Days_To_Deadline  5000 non-null   int64
 4   Priority_Label    5000 non-null   str  
dtypes: int64(2), str(3)
memory usage: 195.4 KB


In [7]:
train_df.isnull().sum()

In [8]:
plt.figure(figsize=(8,5))
sns.countplot(x='Priority_Label', data=train_df, order=['Low','Medium','High'])
plt.title("Priority Distribution")
plt.show()

In [9]:
plt.figure(figsize=(12,6))
sns.countplot(y='Department', data=train_df, order=train_df['Department'].value_counts().index)
plt.title("Department Wise Queries")
plt.show()

In [10]:
plt.figure(figsize=(8,4))
sns.histplot(train_df['Days_To_Deadline'], bins=20)
plt.title("Days To Deadline Distribution")
plt.show()

## 4. Text Cleaning

In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['Clean_Query'] = train_df['Student_Query'].apply(clean_text)
test_df['Clean_Query'] = test_df['Student_Query'].apply(clean_text)

## 5. Train/Test Overlap Check
**This check was missing in the original notebook.** A high overlap between train and test query text inflates accuracy artificially. If the warning below fires, treat the reported accuracy with caution.

In [14]:
overlap = set(train_df['Clean_Query']) & set(test_df['Clean_Query'])
overlap_pct = len(overlap) / max(len(test_df), 1) * 100
print(f"Exact text overlap between train/test: {len(overlap)} rows ({overlap_pct:.1f}% of test set)")
if overlap_pct > 30:
    print("WARNING: High train/test overlap. Accuracy is likely inflated. "
          "Validate on genuinely unseen queries before trusting this number.")

Exact text overlap between train/test: 120 rows (12.0% of test set)


## 6. Train Priority-Prediction Model

In [16]:
X_train = train_df['Clean_Query']
y_train = train_df['Priority_Label']
X_test = test_df['Clean_Query']
y_test = test_df['Priority_Label']

In [17]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        max_depth=12,
        random_state=42
    ))
])

model.fit(X_train, y_train)

In [18]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 100.0 %


In [19]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

        High       1.00      1.00      1.00       313
         Low       1.00      1.00      1.00       350
      Medium       1.00      1.00      1.00       337

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000



In [20]:
cm = confusion_matrix(y_test, predictions)
labels = sorted(y_train.unique())

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## 7. Department Routing & Auto-Response
**Fixed:** previously, `route_department()` and `generate_auto_response()` used two different, hand-duplicated keyword lists that could disagree. Now there is a single rule list (`DEPARTMENT_RULES`), checked with word-boundary regex so `"exam"` won't accidentally match unrelated words like `"example"`.

In [22]:
DEPARTMENT_RULES = [
    (r"\bhall ticket\b", "Examination Cell"),
    (r"\badmit card\b", "Examination Cell"),
    (r"\bexam(ination)?\b", "Examination Cell"),
    (r"\bresult\b", "Examination Cell"),
    (r"\bscholarship\b", "Finance Office"),
    (r"\bfee\b|\bpayment\b", "Finance Office"),
    (r"\bhostel\b", "Hostel Office"),
    (r"\bwifi\b|\binternet\b", "IT Support"),
    (r"\bportal\b|\bpassword\b", "IT Support"),
    (r"\blibrary\b", "Library"),
    (r"\badmission\b", "Admission Cell"),
    (r"\bplacement\b|\binternship\b", "Placement Cell"),
]

RESPONSE_TEMPLATES = {
    "Examination Cell": "Your examination-related query has been forwarded to the Examination Cell. You will receive an update shortly.",
    "Finance Office": "Your fee/scholarship-related query has been registered. Please keep your payment receipt for verification.",
    "Hostel Office": "Your hostel-related complaint has been forwarded to the Hostel Office. Necessary action will be taken soon.",
    "IT Support": "Your portal/internet issue has been sent to the IT Support team. Please allow some time for troubleshooting.",
    "Library": "Your library-related query has been forwarded to the Library Department.",
    "Admission Cell": "Your admission-related query has been registered successfully.",
    "Placement Cell": "Your placement/internship-related query has been forwarded to the Placement Cell.",
    "Administration": "Thank you for contacting the University Helpdesk. Your query has been registered and will be reviewed soon.",
}

def route_department(query):
    q = query.lower()
    for pattern, dept in DEPARTMENT_RULES:
        if re.search(pattern, q):
            return dept
    return "Administration"

def generate_auto_response(department):
    return RESPONSE_TEMPLATES.get(department, RESPONSE_TEMPLATES["Administration"])

## 8. Ticket Generation & Summary

In [24]:
def generate_ticket():
    return "TKT" + str(random.randint(1000, 9999))

def summarize_query(query, max_words=8):
    words = query.split()
    if len(words) <= max_words:
        return query
    return " ".join(words[:max_words]) + "..."

## 9. Full Chatbot (Now Actually Uses the Trained Model)
**This is the main fix.** In the original notebook, the trained `model` was never called inside the chatbot -- priority was simply missing from its output. Here, `predict_priority()` calls `model.predict()` on the cleaned query, and the result is included in the chatbot's response.

In [26]:
def predict_priority(query):
    cleaned = clean_text(query)
    pred = model.predict([cleaned])[0]
    return pred

def university_chatbot(query):
    if not isinstance(query, str) or not query.strip():
        print("Please enter a valid, non-empty query.")
        return None

    ticket_id = generate_ticket()
    department = route_department(query)
    priority = predict_priority(query)
    response = generate_auto_response(department)
    summary = summarize_query(query)

    result = {
        "ticket_id": ticket_id,
        "query": query,
        "summary": summary,
        "department": department,
        "priority": priority,
        "response": response,
    }

    print("=" * 70)
    print("TICKET ID          :", ticket_id)
    print("STUDENT QUERY      :", query)
    print("QUERY SUMMARY      :", summary)
    print("ASSIGNED DEPARTMENT:", department)
    print("PREDICTED PRIORITY :", priority)
    print("AUTO RESPONSE      :", response)
    print("=" * 70)

    return result

## 10. Test the Chatbot

In [28]:
university_chatbot("I am unable to submit my examination form")

TICKET ID          : TKT6154
STUDENT QUERY      : I am unable to submit my examination form
QUERY SUMMARY      : I am unable to submit my examination form
ASSIGNED DEPARTMENT: Examination Cell
PREDICTED PRIORITY : Medium
AUTO RESPONSE      : Your examination-related query has been forwarded to the Examination Cell. You will receive an update shortly.


In [29]:
university_chatbot("My scholarship has not been credited yet")

TICKET ID          : TKT6858
STUDENT QUERY      : My scholarship has not been credited yet
QUERY SUMMARY      : My scholarship has not been credited yet
ASSIGNED DEPARTMENT: Finance Office
PREDICTED PRIORITY : Medium
AUTO RESPONSE      : Your fee/scholarship-related query has been registered. Please keep your payment receipt for verification.


In [30]:
# Off-topic query -> should fall back to Administration, not crash or mis-route
university_chatbot("Can you tell me about the campus annual day event")

TICKET ID          : TKT1361
STUDENT QUERY      : Can you tell me about the campus annual day event
QUERY SUMMARY      : Can you tell me about the campus annual...
ASSIGNED DEPARTMENT: Administration
PREDICTED PRIORITY : Low
AUTO RESPONSE      : Thank you for contacting the University Helpdesk. Your query has been registered and will be reviewed soon.


In [31]:
# Edge case: empty input is now handled gracefully instead of breaking
university_chatbot("")

Please enter a valid, non-empty query.


## 11. Analytics

In [33]:
train_df['Department'].value_counts().plot(kind='bar', figsize=(10,5))
plt.title("Department Analytics")
plt.ylabel("Number of Queries")
plt.show()

In [34]:
train_df['Priority_Label'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Priority Label Distribution")
plt.ylabel('')
plt.show()

In [35]:
from collections import Counter

words = " ".join(train_df['Clean_Query'])
top_words = Counter(words.split())
print(top_words.most_common(20))

[('the', 2100), ('i', 2089), ('my', 1639), ('is', 1396), ('sirmadam', 1029), ('for', 1023), ('hi', 1022), ('hello', 987), ('exam', 969), ('and', 959), ('respected', 954), ('sir', 954), ('how', 863), ('to', 850), ('what', 675), ('not', 652), ('portal', 644), ('fee', 629), ('in', 624), ('need', 601)]
